Assignment No. 4

Title:
Implementation of CUDA Programs for Parallel Computing

Problem Statement:
Write CUDA programs for:
1. Addition of two large vectors
2. Matrix multiplication using CUDA C

In [1]:
!pip install git+https://github.com/afnan47/cuda.git

  Cloning https://github.com/afnan47/cuda.git to /tmp/pip-req-build-3erl5wuf
  Running command git clone --filter=blob:none --quiet https://github.com/afnan47/cuda.git /tmp/pip-req-build-3erl5wuf
  Resolved https://github.com/afnan47/cuda.git to commit aac710a35f52bb78ab34d2e52517237941399eff
  Preparing metadata (setup.py) ... done
  Created wheel for NVCCPlugin: filename=NVCCPlugin-0.0.2-py3-none-any.whl size=4290 sha256=cc72080c6263d6a01119bf145319eae57fa592a2f610d5a43361682073adc7d0
  Stored in directory: /tmp/pip-ephem-wheel-cache-c2u9i2cv/wheels/e8/cf/c3/c90ca0d0bba7969f9b8670f5624f76d097123d656355c77053
Successfully built NVCCPlugin


In [4]:
%load_ext nvcc_plugin

The nvcc_plugin extension is already loaded. To reload it, use:
  %reload_ext nvcc_plugin


Vector Addition

In [5]:
%%cu
#include <iostream>
using namespace std;

__global__ void add(int* A, int* B, int* C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int n = 5;
    int A[] = {1,2,3,4,5};
    int B[] = {5,4,3,2,1};
    int C[5];

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, n*sizeof(int));
    cudaMalloc(&d_B, n*sizeof(int));
    cudaMalloc(&d_C, n*sizeof(int));

    cudaMemcpy(d_A, A, n*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, n*sizeof(int), cudaMemcpyHostToDevice);

    add<<<1, n>>>(d_A, d_B, d_C, n);
    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, n*sizeof(int), cudaMemcpyDeviceToHost);

    cout << "Vector Addition Result: ";
    for(int i=0;i<n;i++) cout << C[i] << " ";

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);


    return 0;
}

Vector Addition Result: 6 6 6 6 6 


Matrix Multiplication

In [6]:
%%cu
#include <iostream>
using namespace std;

__global__ void multiply(int* A, int* B, int* C, int N) {
    int row = threadIdx.y;
    int col = threadIdx.x;

    int sum = 0;
    for(int k=0;k<N;k++) {
        sum += A[row*N + k] * B[k*N + col];
    }

    C[row*N + col] = sum;
}

int main() {
    int N = 2;

    int A[] = {1,2,3,4};
    int B[] = {5,6,7,8};
    int C[4];

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, N*N*sizeof(int));
    cudaMalloc(&d_B, N*N*sizeof(int));
    cudaMalloc(&d_C, N*N*sizeof(int));

    cudaMemcpy(d_A, A, N*N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, N*N*sizeof(int), cudaMemcpyHostToDevice);

    dim3 threads(N, N);

    multiply<<<1, threads>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, N*N*sizeof(int), cudaMemcpyDeviceToHost);

    cout << "Matrix Multiplication Result:\n";
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            cout << C[i*N + j] << " ";
        }
        cout << endl;
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Matrix Multiplication Result:
19 22 
43 50 

